# **Check Datasets**

Check the dataset Rossmann Store Sales

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
info_data = pd.read_csv("store.csv")
print(info_data.head())

In [ ]:
print(info_data.shape)

In [ ]:
print(info_data.info())

Check the dataset used for training

In [ ]:
train_data = pd.read_csv("/kaggle/input/datasets/pratyushakar/rossmann-store-sales/train.csv" ,  low_memory=False)
print(train_data.head())

In [ ]:
print(train_data.shape)

In [ ]:
print(train_data.info())

Concatenate the two datasets to form a complete dataset

In [ ]:
my_data = pd.merge( info_data , train_data , on = 'Store' , how = 'left')
my_data = my_data.sort_values(by = ['Store','Date'] , ascending = True)
print(my_data.head())

In [ ]:
print(my_data.shape)

# **Exploratory Data Analysis** 

**1.Data profilling**

Check information data

In [ ]:
print(my_data.info())

In [ ]:
print(my_data.describe())

check missing values

In [ ]:
missing_values = my_data.isnull().sum()
print(missing_values)

Since no feature has an excessive amount of missing data, we proceed to impute missing values ​​for all columns

In [ ]:
my_data['CompetitionDistance'] = my_data['CompetitionDistance'].fillna(999999)
data_fill_zero = ['CompetitionOpenSinceMonth' , 'CompetitionOpenSinceYear' , 'Promo2SinceWeek' , 'Promo2SinceYear']
my_data[data_fill_zero] = my_data[data_fill_zero].fillna(0)
my_data['PromoInterval'] = my_data['PromoInterval'].fillna('missing')
new_missing_values = my_data.isnull().sum()
print(new_missing_values)

The data has been completely cleaned!

**2.Rule Cleaning**

proceed to remove values likely to cause interference. We remove the 'open' column because if a store is closed, its revenue is zero; since closed days are excluded, the 'open' column would only contain the value 1, rendering it useless for training the model.

In [ ]:
my_data = my_data.drop('Open' , axis = 1)
my_data = my_data[my_data['Sales'] > 0 ]
print('Complete!')

**3.Univariate Analysis**

Check the distribution of the target variable (sales).

In [ ]:
plt.figure(figsize = ( 8 , 5 ))
sns.histplot( data = my_data['Sales'] , kde=True , bins=100)
plt.title('The distribution of the target variable (sales)')
plt.tight_layout()

Since the distribution is heavily skewed to the right, we need to normalize the target variable

In [ ]:
my_data['Sales'] = np.log1p(my_data['Sales'])
plt.figure(figsize = ( 8 , 5 ))
sns.histplot( data = my_data['Sales'] , kde=True , bins = 100)
plt.title('The distribution of the target variable (sales) after casting the log')
plt.tight_layout()

Great! The distribution of the target variable has been transformed into an ideal form (a bell-shaped distribution)

**4.Bivariate Analysis**

4.1 Visualize numrical features correlations

In [ ]:
numrical = my_data.select_dtypes(exclude = 'object' )
numrical_corr = numrical.corr()
plt.figure(figsize= ( 8 , 5))
sns.heatmap( numrical_corr , annot = True , fmt = '.2f' , cmap = 'coolwarm')
plt.title('Visualize feature correlations')

4.2 Visualize categorical features correlations

In [ ]:
my_data['RealSales'] = np.expm1(my_data['Sales'])
fig, axes = plt.subplots( nrows = 2 , ncols = 2 , figsize = ( 12 , 10 ))
axes = axes.flatten()
sns.barplot( x = 'StoreType' , y = 'RealSales' , data = my_data , ax = axes[0])
axes[0].set_title('Revenue from all types of stores')
sns.barplot( x = 'Assortment' , y = 'RealSales' , data = my_data , ax = axes[1] )
axes[1].set_title('Revenue by assortment')
sns.barplot( x = 'PromoInterval' , y = 'RealSales' , data = my_data , ax = axes[2])
axes[2].set_title('Revenue by PromoInterval')
sns.barplot( x = 'StateHoliday' , y = 'RealSales' , data = my_data , ax = axes[3] )
axes[3].set_title('Revenue by StateHoliday')
plt.tight_layout()
my_data = my_data.drop('RealSales' , axis = 1)

In conclusion: Chart 1 shows a customer preference for Store Type B, while Chart 2 indicates a preference for Product Category B. Chart 3 reveals that revenue is higher when no discount code is applied compared to when one is; this is because high-revenue stores do not offer discounts—reserving them instead for lower-revenue stores—rather than the discounts themselves causing a drop in patronage. Finally, Chart 4 displays store revenue on major holidays. Since the "closed" category was excluded, stores that remained open generated significantly higher revenue than on ordinary days; however, because only a few stores were open, the margin of error for columns a, b, and c is extremely high.

4.3 Use Random Forest to determine which features are essential for the machine learning model and categorical data in my_data

Create training and validation sets

In [ ]:
X_train = my_data[(my_data['Date'] < '2013-09-01') & (my_data['Date'] > '2013-01-01') ]
y_train = X_train['Sales']
X_train = X_train.drop('Sales' , axis = 1 )
X_valid = my_data[(my_data['Date'] >='2013-09-01') & (my_data['Date'] < '2014-01-01') ]
y_valid = X_valid['Sales']
X_valid = X_valid.drop('Sales' , axis = 1 )

Setup pipeline

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import category_encoders as ce
numrical_cols = X_train.select_dtypes(exclude = 'object').columns
categorical_cols = X_train.select_dtypes(include = 'object').columns
numrical = Pipeline(steps = [('impute' , SimpleImputer(strategy = 'constant' , fill_value = 0))])
categorical = Pipeline(steps = [('impute' , SimpleImputer(strategy = 'constant' , fill_value = 'missing')) ,
                                ('Target_encoders' , ce.TargetEncoder(smoothing = 10.0))])
preprocessing = ColumnTransformer( transformers = [('num' , numrical , numrical_cols) ,
                                                 ('cate' , categorical , categorical_cols)])
print('Create pipeline complete!')

Create Random Forest model and link model with pipeline

In [ ]:
from sklearn.ensemble import RandomForestRegressor
X_train[categorical_cols] = X_train[categorical_cols].astype(str)
model_test = RandomForestRegressor( n_estimators = 100 , random_state = 0 , n_jobs=-1, max_depth=20 , verbose = False)
model_pipeline = Pipeline ( steps = [('preprocessing' , preprocessing) ,
                                     ('model' , model_test)])
model_pipeline = model_pipeline.fit(X_train , y_train)
print('complete train model')

Evaluate which features are truly useful

In [ ]:
result_step = model_pipeline.named_steps['model']
score = result_step.feature_importances_
cat_features_new =  model_pipeline.named_steps['preprocessing'].named_transformers_['cate'].named_steps['Target_encoders'].get_feature_names_out(categorical_cols)
all_feature_names = list(numrical_cols) + list(cat_features_new)
result_df = pd.DataFrame({'Features': all_feature_names, 'Score': score}).sort_values(by='Score', ascending=False)
plt.figure(figsize=(10, 8))
sns.barplot(x='Score', y='Features', data=result_df.head(15))
plt.title('The impact of top 15 features on sales')
plt.tight_layout()
plt.show()

Console: The columns with low correlation PromoInterval and Promo2SinceYear need to be removed. Regarding the Assortment columns: although the correlations forAssortment a and Assortment c are low, they belong to the same feature as Assortment b (which shows high correlation), so the Assortment column should be retained.

**5. Mutual Information**

Set up the data required to find mutual information.

In [ ]:
X = my_data[(my_data['Date'] < '2013-09-01') & (my_data['Date'] > '2013-01-01') ]
y = X['Sales']
X = X.drop(['Sales', 'Date'] , axis = 1 )
for colname in X.select_dtypes('object'):
    X[colname] , _ = X[colname].factorize()
discrete_features = X.dtypes == int
print('complete!')

check data reciprocity

In [ ]:
from sklearn.feature_selection import mutual_info_regression
mi_scores = mutual_info_regression(X , y , discrete_features = discrete_features)
mi_scores = pd.Series( mi_scores , name = 'MI Scores' , index = X.columns)
mi_scores = mi_scores.sort_values( ascending = False)
mi_scores[::3]

Visualize MI scores on a bar plot

In [ ]:
plt.figure( dpi=100 , figsize = ( 8 , 5 ))
scores = mi_scores.sort_values(ascending = True )
width = np.arange(len(scores))
ticks = list(scores.index)
plt.barh(width , scores)
plt.yticks(width , ticks)
plt.title('Mutual Information Scores')
plt.tight_layout()

Console: Here, we observe a column representing customer volume with unusually high values,since customer numbers are directly proportional to revenue, we should remove the customer column but retain the remaining ones. Even if their individual correlation scores are low, combining them with other variables enhances their value within the predictive model

**6.Feature Selection**

We begin by removing noisy and unnecessary features (such as identifiers).

In [ ]:
my_data = my_data.drop(['PromoInterval' , 'Customers'] , axis = 1 )
print(my_data.head())

# **Feature Enginnering**

**1. create new features useful for predict model**

We start by splitting the Date column so the model can better understand the date information. Later, we need to remove Date columns because it is no longer useful

In [ ]:
my_data['Date'] = pd.to_datetime( my_data['Date'])
my_data['Day'] = my_data['Date'].dt.day
my_data['Month'] = my_data['Date'].dt.month
my_data['Year'] = my_data['Date'].dt.year
my_data = my_data.drop('Date' , axis = 1 )
print(my_data.head())

Create an additional feature regarding competitors' store operating hours so the model can identify revenue trends in the presence of competing stores. Later , we proceed remove CompetitionOpenSinceMonth , CompetitionOpenSinceYear because it is not longer useful

In [ ]:
my_data['CompetitorLongevity'] = my_data['Year'] - my_data['CompetitionOpenSinceYear']
my_data.loc[my_data['CompetitionOpenSinceYear'] == 0, 'CompetitorLongevity'] = 0
my_data = my_data.drop([ 'CompetitionOpenSinceMonth' , 'CompetitionOpenSinceYear'] , axis = 1)
print(my_data.head())

**2. Time Series processing**

2.1 Trend feature

smooth the data and observe long-term trends

In [ ]:
my_data['Date'] = pd.to_datetime(my_data[['Year', 'Month', 'Day']])
daily_sales = my_data.groupby('Date')['Sales'].mean()
moving_average = daily_sales.rolling(
    window = 365 ,
    center = True ,
    min_periods = 183
).mean()
ax = daily_sales.plot(style=".", color="0.5")
moving_average.plot(
    ax=ax, linewidth=3, title="Rossmann store sales - 365-Day Moving Average", legend=False,
);

The moving average does not move sideways or trend downward but instead rises steadily over time remaining largely unaffected by noise indicating that the chain of stores is on a growth trajectory. Meanwhile, the uniform dispersion of data suggests a highly consistent business operational pattern across the years.

engineer additional trend-related data features

In [ ]:
from statsmodels.tsa.deterministic import DeterministicProcess
daily_sales.index = daily_sales.index.to_period('D')
dp = DeterministicProcess(
    index = daily_sales.index ,
    constant = True ,
    order = 1 ,
    drop = True
)
my_data['Date'] = my_data['Date'].dt.to_period('D')
X_trend = dp.in_sample()
print(X_trend.head())

visualize long-term trends

In [ ]:
from sklearn.linear_model import LinearRegression
model = LinearRegression(fit_intercept=False)
y = daily_sales 
model.fit(X_trend,y)
y_pred = pd.Series(model.predict(X_trend), index=X_trend.index)
plt.figure(figsize=(10, 5))
ax = y.plot(style=".", color="0.5", alpha=0.5, label="Actual Sales")
y_pred.plot(ax=ax, linewidth=3, color="blue", label="Linear trend")
plt.title("Rossmann Store Sales - Evaluation Linear Regression model")
plt.legend()
plt.show()

Try forecasting revenue for the next 30 days

In [ ]:
X_trend = dp.out_of_sample(steps=30)
y_fore = pd.Series(model.predict(X_trend), index=X_trend.index)
plt.figure(figsize=(12, 5))
ax = y.plot(style=".-", color="0.5", alpha=0.6, label="Sales")
y_pred.plot(ax=ax, linewidth=3, color="blue", label="Trend")
y_fore.plot(ax=ax, linewidth=3, color="red", label="Trend Forecast")
plt.title("Daily sales - Linear Trend Forecast")
plt.legend()
plt.show()

2.2 Seasonality

Plot a chart showing the impact of dates on revenue.

In [ ]:
def seasonal_plot( X , y , period , freq , ax=None ):
    if ax is None:
        fig, ax = plt.subplots()
    sns.lineplot( data = X , x = freq , y = y , hue = period , errorbar = None , palette="husl" , ax = ax , legend = False)
    ax.set_title(f'Seasonal Plot ({period}/{freq})')
    return ax
X_season = daily_sales.to_frame(name='Sales')
X_season['day'] = X_season.index.dayofweek
X_season['week'] = X_season.index.week
X_season['dayofyear'] = X_season.index.dayofyear
X_season['year'] = X_season.index.year
fig, (ax0 , ax1) = plt.subplots(2 , 1 , figsize = (11 , 6))
seasonal_plot(X_season , y = 'Sales' , period = 'week' , freq = 'day' , ax = ax0 )
seasonal_plot(X_season , y = 'Sales' , period = 'year' , freq = 'dayofyear' , ax = ax1 )
plt.tight_layout()

The chart clearly illustrates the cyclical nature of the store chain's revenue: sales peak on Sundays and drop sharply on Saturdays, while remaining relatively stable throughout the rest of the week

Now let's look at the periodogram:

In [ ]:
from scipy.signal import periodogram
def plot_periodogram(ts, detrend='linear', ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 5))
    frequencies, variance = periodogram( ts, detrend=detrend, scaling='spectrum')
    ax.step(frequencies, variance, color="purple")
    ax.set_title('periodogram - Rossmann Store Sales')
    periods = [365, 30, 7, 3.5]
    x_ticks = [1 / p for p in periods]
    x_labels = ['Annual (1)', 'Monthly (12)', 'Weekly (52)', 'Semiweekly (104)']
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(x_labels, rotation=30)
    return ax
plot_periodogram( X_season['Sales'] )

create seasonality feature 

In [ ]:
from statsmodels.tsa.deterministic import CalendarFourier
fourier_annual = CalendarFourier(freq = 'YE' , order = 10 )
fourier_month = CalendarFourier(freq = 'W' , order = 2 )
dp = DeterministicProcess(
    index = X_season.index ,
    constant = True ,
    order = 1,
    seasonal = True ,
    additional_terms = [ fourier_annual , fourier_month ] ,
    drop = True
)
X_fourier = dp.in_sample()
print(X_fourier.head())

Visualize seasonality and attempt to forecast sales for the next 30 days

In [ ]:
from sklearn.linear_model import Lasso
model_lasso = Lasso(alpha = 1.0)
model.fit(X_fourier , y)
y_pred = pd.Series( model.predict(X_fourier) , index = X_fourier.index)
X_for = dp.out_of_sample(steps = 30)
y_fore = pd.Series( model.predict(X_for) , index = X_for.index)
plt.figure(figsize=(12, 5))
ax = y.plot(style=".-", color="0.5", alpha=0.7, label="Sales")
y_pred.plot(ax=ax, linewidth=1, color="blue", label="Season")
y_fore.plot(ax=ax, linewidth=1, color="red", label="Season Forecast")
plt.title("Daily sales - Linear Season Forecast")
plt.legend()

Add trend and fourier feature to the data table

In [ ]:
my_data = pd.merge( my_data , X_fourier , how = 'left' , on = 'Date' )
print(my_data.head())

4.3 Create time series feature

Revenue adjusted for trend and seasonal factors

In [ ]:
y_deseason = y - y_pred 
y_deseason.name = 'Sale Deseasoned'
ax = y_deseason.plot()
ax.set_title('Sales of Rossman Store Sales(Deseasoned)')

When unaffected by seasonality, the data is driven solely by random noise or promotions.

perform weekly data smoothing with y

In [ ]:
y_ma = y.rolling( window = 7 , center = True ).mean()
ax = y_ma.plot()
ax.set_title('Seven day moving average')

The moving average plot exhibits significant amplitude, clearly reflecting annual revenue surge cycles, whereas the residual plot shows reduced amplitude; thus, we conclude that the model captures over 80% of the data variation, with the remaining residuals representing only outliers and noise

Read the remaining rules using lag

In [ ]:
from statsmodels.graphics.tsaplots import plot_pacf
fig, ax = plt.subplots(figsize=(10, 4))
plot_pacf(y_deseason, lags=8, ax=ax);

In [ ]:
def my_plot_lags ( y , lags = 8 , nrows = 2 ):
    ncols = lags//nrows
    fig , axes = plt.subplots( nrows = nrows , ncols = ncols , figsize = (16 , 6) , sharex = True , sharey = True )
    axes = axes.flatten()
    for i in range(1 , lags + 1):
        ax = axes[i-1]
        y_lag = y.shift(i)
        sns.regplot( x = y_lag, y = y , ax = ax , scatter_kws = {'s': 5 , 'alpha': 0.7 } ,line_kws={'color': 'red'}, lowess=True )
        ax.set_title ( f'Lag {i}')
        ax.set_xlabel ('Sales deseasoned')
        if i % ncols == 1:
            ax.set_ylabel('sales_deseasoned')
        else:
            ax.set_ylabel('')
    plt.tight_layout()
my_plot_lags( y_deseason , lags = 8 , nrows = 2)

The Partial Autocorrelation plot shows that most lags exceed the blue threshold. however, this is due to the narrow error margin rather than the lags actually being useful. Upon closer inspection of the individual lags, most appear flat or fluctuate in response to noise, whereas only lag 1 and lag 8 prove useful for prediction being relatively unaffected by noise so we proceed to add these lags to my_data.

In [ ]:
my_lags = pd.DataFrame(index=y_deseason.index)
my_lags['lag_1_resid'] = y_deseason.shift(1).values
my_lags['lag_8_resid'] = y_deseason.shift(8).values
my_data['lag_1'] = my_data.groupby('Store')['Sales'].shift(1)
my_data['lag_8'] = my_data.groupby('Store')['Sales'].shift(8)
my_lags = my_lags.reset_index()
if 'index' in my_lags.columns:
    my_lags = my_lags.rename(columns={'index': 'Date'})
my_data = pd.merge(my_data, my_lags, how='left', on='Date')
my_data = my_data.dropna()
print(my_data[['Date', 'Store', 'lag_1', 'lag_8' , 'lag_1_resid' , 'lag_8_resid']].head())

**3. Process data to train the model**

3.1 preparing data for the hybrid model

After deriving the patterns regarding seasonal trends, the 'Date' column is no longer necessary, so we remove it; at the same time, we delete any empty columns that introduce noise into the data.

In [ ]:
my_data = my_data.drop('Date' , axis = 1 )
my_data = my_data.dropna()
print(my_data.head())

Create Boosted Hybrid class and related functions.

In [ ]:
from sklearn.base import BaseEstimator
class BoostedHybrid(BaseEstimator):
    def __init__( self , model_1 , model_2 ):
        self.model_1 = model_1 
        self.model_2 = model_2
        self.y_columns = None
def fit( self , X_1 , X_2 , y ):
    self.model_1.fit(X_1 , y)
    y_pred = self.model_1.predict(X_1)
    y_fit = pd.DataFrame( y_pred , index = X_1.index , columns = y.columns )
    y_resid = y - y_fit
    y_resid = y_resid.stack().squeeze()
    self.model_2.fit(X_2 , y_resid)
    self.y_columns = y.columns
    self.y_fit = y_fit
    self.y_resid = y_resid
    return self
BoostedHybrid.fit = fit
def predict( self , X_1 , X_2 ):
    y_pred = pd.DataFrame( self.model_1.predict(X_1) , index = X_1.index , columns = self.y_columns )
    y_pred = y_pred.stack().squeeze()
    y_pred += self.model_2.predict(X_2)
    return y_pred.unstack()
BoostedHybrid.predict = predict
print('BoostHybrid class was completed!')

Create a dataset for training machine learning models.

In [ ]:
feature_hd = ['Store' , 'StoreType' , 'Assortment' , 'CompetitionDistance' , 'CompetitorLongevity' , 'Promo2', 'Promo2SinceWeek' , 'Promo2SinceYear' , 'DayOfWeek' , 'Promo' , 'StateHoliday' , 'SchoolHoliday' , 'Day' , 'Month' , 'lag_1_resid' , 'lag_8_resid' , 'Year']
mask_train = (my_data['Year'] < 2015)
X_train_hd = my_data[mask_train]
X_valid_hd = my_data[(my_data['Year'] == 2015) & (my_data['Month'] >= 1) & (my_data['Month'] < 5)]
X_test_hd = my_data[(my_data['Year'] == 2015) & (my_data['Month'] >= 5)]
y_train_hd = X_train_hd[['Sales']]
X_train_hd = X_train_hd.drop('Sales' , axis = 1 )
y_valid_hd = X_valid_hd[['Sales']]
X_valid_hd = X_valid_hd.drop('Sales' , axis = 1 )
y_test_hd = X_test_hd[['Sales']]
X_test_hd = X_test_hd.drop('Sales' , axis = 1 )
X_test_1 = X_test_hd.drop( feature_hd , axis = 1 )
X_test_2 = X_test_hd[feature_hd]
X_train_1 = X_train_hd.drop( feature_hd, axis = 1 )
X_train_2 = X_train_hd[feature_hd]
X_valid_1 = X_valid_hd.drop( feature_hd , axis = 1 )
X_valid_2 = X_valid_hd[feature_hd]
print('setup data set complete!')

Handle categorical features using target_encoder.

In [ ]:
target_encoder = ce.TargetEncoder(smoothing = 10.0)
X_train_2 = target_encoder.fit_transform(X_train_2 , y_train_hd)
X_valid_2 = target_encoder.transform(X_valid_2)
X_test_2 = target_encoder.transform(X_test_2)
print('Complete!')

3.2 preparing data for a single LGBM model

Create train model data

In [ ]:
mask_train = (my_data['Year'] < 2015) | ((my_data['Year'] == 2015) & (my_data['Month'] < 2))
X_train = my_data[mask_train]
X_valid = my_data[(my_data['Year'] == 2015) & (my_data['Month'] >= 1) & (my_data['Month'] < 5)]
X_test = my_data[(my_data['Year'] == 2015) & (my_data['Month'] >= 5) ]
y_train = X_train['Sales']
X_train = X_train.drop('Sales' , axis = 1 )
y_valid = X_valid['Sales']
X_valid = X_valid.drop('Sales' , axis = 1 )
y_test = X_test['Sales']
X_test = X_test.drop('Sales' , axis = 1 )
feature_lgb = ['Store' , 'StoreType' , 'Assortment' , 'CompetitionDistance' , 'CompetitorLongevity' , 'Promo2', 'Promo2SinceWeek' , 'Promo2SinceYear' , 'DayOfWeek' , 'Promo' , 'StateHoliday' , 'SchoolHoliday' , 'Day' , 'Month' , 'lag_1' , 'lag_8' , 'Year']
X_train = X_train[feature_lgb]
X_valid = X_valid[feature_lgb]
X_test = X_test[feature_lgb]
print('setup data set complete!')

processing data

In [ ]:
X_train = target_encoder.fit_transform(X_train, y_train)
X_valid = target_encoder.transform(X_valid)
X_test = target_encoder.transform(X_test)
print('complete!')

**4. Data clustering**

Select features for clustering

In [ ]:
features_1 = ['CompetitionDistance' , 'CompetitorLongevity']
X_train_scale_1 = X_train.loc[: , features_1]
X_valid_scale_1 = X_valid.loc[: , features_1]
X_test_scale_1 = X_test.loc[: , features_1]
X_train_2_scale_1 = X_train_2.loc[: , features_1]
X_valid_2_scale_1 = X_valid_2.loc[: , features_1]
X_test_2_scale_1 = X_test_2.loc[: , features_1]
features_2 = ['StoreType' , 'Assortment']
X_train_scale_2 = X_train.loc[: , features_2]
X_valid_scale_2 = X_valid.loc[: , features_2]
X_test_scale_2 = X_test.loc[: , features_2]
X_train_2_scale_2 = X_train_2.loc[: , features_2]
X_valid_2_scale_2 = X_valid_2.loc[: , features_2]
X_test_2_scale_2 = X_test_2.loc[: , features_2]
print('complete!')

normalize parameters for clustering

In [ ]:
from sklearn.preprocessing import StandardScaler
scale_1 = StandardScaler().set_output(transform="pandas")
scale_2 = StandardScaler().set_output(transform="pandas")
scale_3 = StandardScaler().set_output(transform="pandas")
scale_4 = StandardScaler().set_output(transform="pandas")
X_train_scale_1 = scale_1.fit_transform(X_train_scale_1)
X_valid_scale_1 = scale_1.transform(X_valid_scale_1)
X_test_scale_1 = scale_1.transform(X_test_scale_1)
X_train_2_scale_1 = scale_3.fit_transform(X_train_2_scale_1)
X_valid_2_scale_1 = scale_3.transform(X_valid_2_scale_1)
X_test_2_scale_1 = scale_3.transform(X_test_2_scale_1)
X_train_scale_2 = scale_2.fit_transform(X_train_scale_2)
X_valid_scale_2 = scale_2.transform(X_valid_scale_2)
X_test_scale_2 = scale_2.transform(X_test_scale_2)
X_train_2_scale_2 = scale_4.fit_transform(X_train_2_scale_2)
X_valid_2_scale_2 = scale_4.transform(X_valid_2_scale_2)
X_test_2_scale_2 = scale_4.transform(X_test_2_scale_2)
print('complete!')

Using K mean algorithm to clusters features

In [ ]:
from sklearn.cluster import KMeans
kmean_1 = KMeans( n_clusters = 5 , n_init = 5 , random_state = 0)
kmean_2 = KMeans( n_clusters = 5 , n_init = 5 , random_state = 0)
kmean_3 = KMeans( n_clusters = 5 , n_init = 5 , random_state = 0)
kmean_4 = KMeans( n_clusters = 5 , n_init = 5 , random_state = 0)
X_train['Cluster_1'] = kmean_1.fit_predict(X_train_scale_1)
X_valid['Cluster_1'] = kmean_1.predict(X_valid_scale_1)
X_test['Cluster_1'] = kmean_1.predict(X_test_scale_1)
X_train['Cluster_1'] = X_train['Cluster_1'].astype('string')
X_valid['Cluster_1'] = X_valid['Cluster_1'].astype('string')
X_test['Cluster_1'] = X_test['Cluster_1'].astype('string')
X_train['Cluster_2'] = kmean_2.fit_predict(X_train_scale_2)
X_valid['Cluster_2'] = kmean_2.predict(X_valid_scale_2)
X_test['Cluster_2'] = kmean_2.predict(X_test_scale_2)
X_train['Cluster_2'] = X_train['Cluster_2'].astype('string')
X_valid['Cluster_2'] = X_valid['Cluster_2'].astype('string')
X_test['Cluster_2'] = X_test['Cluster_2'].astype('string')
X_train_2['Cluster_1'] = kmean_3.fit_predict(X_train_2_scale_1)
X_valid_2['Cluster_1'] = kmean_3.predict(X_valid_2_scale_1)
X_test_2['Cluster_1'] = kmean_3.predict(X_test_2_scale_1)
X_train_2['Cluster_1'] = X_train_2['Cluster_1'].astype('string')
X_valid_2['Cluster_1'] = X_valid_2['Cluster_1'].astype('string')
X_test_2['Cluster_1'] = X_test_2['Cluster_1'].astype('string')
X_train_2['Cluster_2'] = kmean_4.fit_predict(X_train_2_scale_2)
X_valid_2['Cluster_2'] = kmean_4.predict(X_valid_2_scale_2)
X_test_2['Cluster_2'] = kmean_4.predict(X_test_2_scale_2)
X_train_2['Cluster_2'] = X_train_2['Cluster_2'].astype('string')
X_valid_2['Cluster_2'] = X_valid_2['Cluster_2'].astype('string')
X_test_2['Cluster_2'] = X_test_2['Cluster_2'].astype('string')
cluster_encoder_1 = ce.TargetEncoder(smoothing = 10.0)
cluster_encoder_2 = ce.TargetEncoder(smoothing = 10.0)
cluster_cols = ['Cluster_1' , 'Cluster_2']
X_train[cluster_cols] = cluster_encoder_1.fit_transform(X_train[cluster_cols] , y_train)
X_valid[cluster_cols] = cluster_encoder_1.transform(X_valid[cluster_cols])
X_test[cluster_cols] = cluster_encoder_1.transform(X_test[cluster_cols])
X_train_2[cluster_cols] = cluster_encoder_2.fit_transform( X_train_2[cluster_cols] , y_train_hd)
X_valid_2[cluster_cols] = cluster_encoder_2.transform(X_valid_2[cluster_cols])
X_test_2[cluster_cols] = cluster_encoder_2.transform(X_test_2[cluster_cols])
print('complete!')

# **Setup and process train model machine learning**

**1. Boosted Hybrid Model**

Create a boosted hybrid model for training.

In [ ]:
import lightgbm as lgb
model_hybrid = BoostedHybrid( model_1 = LinearRegression() , model_2 = lgb.LGBMRegressor(n_estimators = 3000 , learning_rate = 0.05 , max_depth = 10 , n_jobs = -1 , num_leaves = 127))
model_hybrid.fit(X_train_1, X_train_2 , y_train_hd )

Then, we will perform forecasting on the validation set.

In [ ]:
y_pred_valid_hd = model_hybrid.predict(X_valid_1 , X_valid_2)
y_pred_valid_hd = y_pred_valid_hd.clip(0.0)
y_pred_test_hd = model_hybrid.predict(X_test_1 , X_test_2)
y_pred_test_hd = y_pred_test_hd.clip(0.0)
y_valid_hd = np.expm1(y_valid_hd)
y_pred_valid_hd = np.expm1(y_pred_valid_hd)
y_test_hd = np.expm1(y_test_hd)
y_pred_test_hd = np.expm1(y_pred_test_hd)

print('complete!')

representing the learning and prediction processes of the hybrid model.

In [ ]:
y_sample = y_train_hd['Sales'][:100].values
y_fit_sample = model_hybrid.y_fit['Sales'][:100].values
y_res = model_hybrid.y_resid.unstack()
y_resid_sample = y_res['Sales'][:100].values
fig, axes = plt.subplots(3, 1, figsize=(15, 12), sharex=True)
axes[0].plot( y_sample , label='Actual Revenue', color='gray', alpha=0.6)
axes[0].plot(y_fit_sample, label='The Projected Revenue Of Model_1', color='red', linewidth=2)
axes[0].set_title('Linear Regressor Line')
axes[0].legend()

axes[1].plot( y_resid_sample ,  label='Unexplained Residual', color='orange')
axes[1].axhline(0, color='black', linestyle='--')
axes[1].set_title('Residual For LGBM Regressor')
axes[1].legend()

y_valid_sample = y_valid_hd[:100].values
y_pred_valid_sample = y_pred_valid_hd[:100].values

axes[2].plot(y_valid_sample, label='Actual(Valid)', color='gray', alpha=0.7)
axes[2].plot(y_pred_valid_sample, label='Forecast from the complete hybrid model', color='green', linewidth=2)
axes[2].set_title('Consolidated revenue forecast based on two models')
axes[2].legend()
plt.tight_layout()

**2. Single LGBM Model**

set up the machine learning model and adjust the necessary parameters of model

In [ ]:
import lightgbm as lgb
model_lgb = lgb.LGBMRegressor( n_estimators = 3000 , learning_rate = 0.05 , max_depth = 10 , n_jobs = -1 , num_leaves = 127 )
model_lgb.fit( X_train, y_train, 
           eval_set=[(X_valid, y_valid)],
           callbacks=[lgb.early_stopping(stopping_rounds=50) , lgb.log_evaluation(period=200)]
)

visualize the model's learning process

In [ ]:
lgb.plot_metric(model_lgb, metric='l2', figsize=(10, 6))
plt.title('Chart of model learning process (Learning Curve)')
plt.xlabel('Number of Trees (Iterations)')
plt.ylabel('Error MSE (l2)')
plt.show()

machine learning model testing

In [ ]:
from sklearn.metrics import r2_score , mean_absolute_error , mean_squared_error
my_pre_val = model_lgb.predict(X_valid)
my_pre_test = model_lgb.predict(X_test)
y_valid = np.expm1(y_valid)
y_test = np.expm1(y_test)
my_pre_val = np.expm1(my_pre_val)
my_pre_test = np.expm1(my_pre_test)
print('Complete!')

the process of learning and prediction on the datasets.

In [ ]:
fig, axes = plt.subplots( nrows = 3 , ncols = 1 , figsize = (15 , 12 ) , sharex = True )
y_train_sample = y_train[:100].values
y_train_pred = model_lgb.predict(X_train)
y_train_pred_sample = y_train_pred[:100]
y_valid_sample = y_valid[:100].values
y_valid_pred_sample = my_pre_val[:100]
y_test_sample = y_test[:100].values
y_test_pred_sample = my_pre_test[:100]
axes[0].plot( y_train_sample , label = 'Actual(Train set)' , color = 'gray' , alpha = 0.7)
axes[0].plot( y_train_pred_sample , label = 'Forecasting of LGBM model on the train set' , color = 'red' , linewidth = 2 )
axes[0].set_title('The training and prediction process of the LGBM model on the training set')
axes[0].legend()

axes[1].plot( y_valid_sample , label = 'Actual(Valid set)' , color = 'gray' , alpha = 0.7)
axes[1].plot( y_valid_pred_sample , label = 'Forecasting of LGBM model on the valid set' , color = 'blue' , linewidth = 2 )
axes[1].set_title('The training and prediction process of the LGBM model on the validition set')
axes[1].legend()

axes[2].plot( y_test_sample , label = 'Actual(Test set)' , color = 'gray' , alpha = 0.7)
axes[2].plot( y_test_pred_sample , label = 'Forecasting of LGBM model on the test set' , color = 'green' , linewidth = 2 )
axes[2].set_title('The training and prediction process of the LGBM model on the test set')
axes[2].legend()
plt.tight_layout()

# **Model Evaluation**

**1.BoostedHybrid Model**

The error between the predicted result and the actual result.

In [ ]:
from sklearn.metrics import mean_absolute_error , mean_squared_error , root_mean_squared_error
Error_hd = {}
Error_hd['mae_val'] = mean_absolute_error( y_valid_hd , y_pred_valid_hd )
Error_hd['mse_val'] = mean_squared_error( y_valid_hd , y_pred_valid_hd )
Error_hd['rmse_val'] = root_mean_squared_error( y_valid_hd , y_pred_valid_hd )
Error_hd['mae_test'] = mean_absolute_error( y_test_hd , y_pred_test_hd )
Error_hd['mse_test'] = mean_squared_error( y_test_hd , y_pred_test_hd )
Error_hd['rmse_test'] = root_mean_squared_error( y_test_hd , y_pred_test_hd )
print('MAE of valid: ' , Error_hd['mae_val'])
print('MSE of valid: ' , Error_hd['mse_val'])
print('RMSE of valid: ' , Error_hd['rmse_val'])
print('MAE of test: ' , Error_hd['mae_test'])
print('MSE of test: ' , Error_hd['mse_test'])
print('RMSE of test: ' , Error_hd['rmse_test'])

Visualizing errors using charts.

In [ ]:
error_hd_df = pd.DataFrame(Error_hd, index=[0])
error_hd_df['method'] = error_hd_df.index
error_hd_df = error_hd_df.melt( id_vars = 'method' , var_name = 'Method_error' , value_name= 'Error')
error_hd_df_clean = error_hd_df[~error_hd_df['Method_error'].str.startswith('mse')]
plt.figure( figsize = ( 8 , 5))
ax = sns.barplot( data = error_hd_df_clean , x = 'Method_error' , y = 'Error' , hue = 'method')
plt.legend(loc = 'lower right')
for container in ax.containers:
    ax.bar_label(container , fmt = '%.1f')

Observations show a significant discrepancy between the error rates on the validation set and the test set. This indicates that the hybrid model is somewhat unstable and not yet fully optimized.

Evaluating the percentage of real-world patterns captured by the model.

In [ ]:
from sklearn.metrics import r2_score
measure_hd = {}
n_valid_hd = len(y_valid_hd)
k_valid_hd = X_valid_1.shape[1] + X_valid_2.shape[1]
n_test_hd = len(y_test_hd)
k_test_hd = X_test_1.shape[1] + X_test_2.shape[1]
measure_hd['r2_score_valid'] = r2_score( y_valid_hd , y_pred_valid_hd)
measure_hd['adj_r2_score_valid'] = 1 - ((1 - measure_hd['r2_score_valid'])*(n_valid_hd -1))/(n_valid_hd - k_valid_hd - 1)
measure_hd['r2_score_test'] = r2_score( y_test_hd , y_pred_test_hd)
measure_hd['adj_r2_score_test'] = 1 - ((1 - measure_hd['r2_score_test'])*(n_test_hd -1))/(n_test_hd - k_test_hd - 1)
print('R-squared of valid: ' , measure_hd['r2_score_valid'] )
print('Adjusted R-squared of valid: ' , measure_hd['adj_r2_score_valid'])
print('R-squared of test: ' , measure_hd['r2_score_test'] )
print('Adjusted R-squared of test: ' , measure_hd['adj_r2_score_test'])

Visualizing errors using charts.

In [ ]:
mea_hd_df = pd.DataFrame(measure_hd , index = [0])
mea_hd_df['method'] = mea_hd_df.index
mea_hd_df = mea_hd_df.melt( id_vars = 'method' , var_name = 'Measure' , value_name= 'Accuracy')
plt.figure( figsize = ( 8 , 5))
ax = sns.barplot( data = mea_hd_df , x = 'Measure' , y = 'Accuracy' , hue = 'method')
plt.legend(loc = 'lower right')
for container in ax.containers:
    ax.bar_label(container)

An analysis of the scores reveals a significant discrepancy between the validation and test sets, underscoring the model's instability. Contrary to typical expectations, validation accuracy is lower than test accuracy due to distinct volatility patterns across the two periods. Specifically, the validation set is highly volatile, whereas the test data is relatively stable. This suggests that while the model is adept at forecasting steady, low-volatility periods, it lacks the robustness to handle sudden, anomalous revenue spikes.

Compare the model's predicted results with the actual results by using scatter plots and histograms to generate residual plots for the training and validation datasets.

In [ ]:
fig , axes = plt.subplots( nrows = 3 , ncols = 1 , figsize = (12,8))
axes.flatten()
sns.histplot( data = y_train_hd - model_hybrid.predict(X_train_1 , X_train_2) , kde = True , ax = axes[0])
axes[0].set_title('The residual plot for the X_train set is represented by a histograms plot')
sns.histplot( data = y_valid_hd - y_pred_valid_hd , kde = True , ax = axes[1])
axes[1].set_title('The residual plot for the X_valid set is represented by a histograms plot')
sns.histplot( data = y_test_hd - y_pred_test_hd , kde = True , ax = axes[2])
axes[2].set_title('The residual plot for the X_test set is represented by a histograms plot')
plt.tight_layout()

In [ ]:
fig , axes = plt.subplots( nrows = 3 , ncols = 1 , figsize = (12,8))
axes.flatten()
sns.residplot( x = y_train_hd , y = model_hybrid.predict(X_train_1 , X_train_2) , ax = axes[0])
axes[0].set_title('The residual plot for the X_train set is represented by a scatter plot')
sns.residplot( x = y_valid_hd , y = y_pred_valid_hd , ax = axes[1])
axes[1].set_title('The residual plot for the X_valid set is represented by a scatter plot')
sns.residplot( x = y_test_hd , y = y_pred_test_hd , ax = axes[2])
axes[2].set_title('The residual plot for the X_test set is represented by a scatter plot')
plt.tight_layout()

Regarding the distribution plot, the majority of the residual data is concentrated around zero, forming a bell-shaped distribution. As for the scatter plot, the residuals are tightly clustered, forming a cloud of data points around the horizontal axis (at zero). so we can conclude that the model is learning effectively and shows little sign of overfitting.

**2. LGBM Model**

Model error, the difference between the predicted price and the actual price.

In [ ]:
Error = {}
Error['mae_val'] = mean_absolute_error(y_valid, my_pre_val)
Error['mse_val'] = mean_squared_error(y_valid, my_pre_val)
Error['rmse_val'] = root_mean_squared_error(y_valid, my_pre_val)
Error['mae_test'] = mean_absolute_error(y_test, my_pre_test)
Error['mse_test'] = mean_squared_error(y_test, my_pre_test)
Error['rmse_test'] = root_mean_squared_error(y_test, my_pre_test)
print('MAE of valid : ' , Error['mae_val'] )
print('MSE of valid: ' , Error['mse_val'] )
print('RMSE of valid: ' , Error['rmse_val'])
print('MAE of test: ' , Error['mae_test'] )
print('MSE of test: ' , Error['mse_test'] )
print('RMSE of test: ' , Error['rmse_test'] )

Visualizing errors using charts

In [ ]:
error_df = pd.DataFrame(Error, index=[0])
error_df['method'] = error_df.index
error_df = error_df.melt( id_vars = 'method' , var_name = 'Method_error' , value_name= 'Error')
error_df_clean = error_df[~error_df['Method_error'].str.startswith('mse')]
plt.figure( figsize = ( 8 , 5))
ax = sns.barplot( data = error_df_clean , x = 'Method_error' , y = 'Error' , hue = 'method')
plt.legend(loc = 'lower right')
for container in ax.containers:
    ax.bar_label(container , fmt = '%.1f')

Great! The discrepancy between the validation and test sets is minimal, leading to the conclusion that the model is not overfitting. Furthermore, the reason the MAE score differs significantly from the RMSE and MSE scores is that the latter two impose heavier penalties on prediction errors (MAE < RMSE < MSE). MSE was excluded from the chart because the magnitude of the error is so large that including it would distort the visualization.

Evaluating the percentage of real-world patterns captured by the model

In [ ]:
from sklearn.metrics import r2_score
measure = {}
n_valid = len(y_valid)
k_valid = X_valid.shape[1]
n_test = len(y_test)
k_test = X_test.shape[1]
measure['r2_score_valid'] = r2_score(y_valid, my_pre_val)
measure['adj_r2_score_valid'] = 1 - ((1-measure['r2_score_valid'])*(n_valid - 1))/(n_valid - k_valid - 1)
measure['r2_score_test'] = r2_score(y_test , my_pre_test)
measure['adj_r2_score_test'] = 1 - ((1-measure['r2_score_test'])*(n_test - 1))/(n_test - k_test - 1)
print('R-squared of valid: ' , measure['r2_score_valid'] )
print('Adjusted R-squared of valid: ' , measure['adj_r2_score_valid'])
print('R-squared of test: ' , measure['r2_score_test'] )
print('Adjusted R-squared of test: ' , measure['adj_r2_score_test'])

Visualizing using charts

In [ ]:
mea_df = pd.DataFrame(measure , index = [0])
mea_df['method'] = mea_df.index
mea_df = mea_df.melt( id_vars = 'method' , var_name = 'Measure' , value_name= 'Accuracy')
plt.figure( figsize = ( 8 , 5))
ax = sns.barplot( data = mea_df , x = 'Measure' , y = 'Accuracy' , hue = 'method')
plt.legend(loc = 'lower right')
for container in ax.containers:
    ax.bar_label(container)

The minimal discrepancy between the validation and test sets indicates that the model is learning effectively and shows little deviation from actual results.

Compare the model's predicted results with the actual results by using scatter plots and histograms to generate residual plots for the training, validation, and test datasets.

In [ ]:
fig, axes = plt.subplots( nrows = 3 , ncols = 1 , figsize = (12 , 8))
axes.flatten()
sns.histplot( data = y_train - model_lgb.predict(X_train) , kde = True , ax = axes[0] )
axes[0].set_title('The residual plot for the X_train set is represented by a histograms plot')
sns.histplot( data = y_valid - my_pre_val , kde = True , ax = axes[1] )
axes[1].set_title('The residual plot for the X_valid set is represented by a histograms plot')
sns.histplot( data = y_test - my_pre_test , kde = True ,  ax = axes[2] )
axes[2].set_title('The residual plot for the X_test set is represented by a histograms plot')
plt.tight_layout()

The residual histograms for the datasets show a bell-shaped distribution that is nearly symmetrical around zero, indicating that the vast majority of the model's predictions are accurate or deviate only minimally.

In [ ]:
fig, axes = plt.subplots( nrows = 3 , ncols = 1 , figsize = (12 , 8))
axes.flatten()
sns.residplot( x = y_train , y = model_lgb.predict(X_train) , ax = axes[0] )
axes[0].set_title('The residual plot for the X_train set is represented by a scatter plot')
sns.residplot( x = y_valid , y = my_pre_val , ax = axes[1] )
axes[1].set_title('The residual plot for the X_valid set is represented by a scatter plot')
sns.residplot( x = y_test , y = my_pre_test , ax = axes[2] )
axes[2].set_title('The residual plot for the X_test set is represented by a scatter plot')
plt.tight_layout()

Across the three datasets, the majority of the blue dots are extremely densely clustered and tightly aligned with the horizontal dotted line at the zero mark. This demonstrates that the model's overall error is very low and that it is reliable. 

**3. Compare between LGBM model and Hybrid model**

conduct a comparison of the error rates of the two models.

In [ ]:
Error_hd_df = pd.DataFrame( Error_hd , index = ['Hybrid'])
error_lgb_df = pd.DataFrame(Error, index=['LGBM'])
Error_models_df = pd.concat([error_lgb_df , Error_hd_df])
Error_models_df['Method'] = Error_models_df.index
Error_models_df = Error_models_df.melt(id_vars = 'Method' , var_name = 'Error Method' , value_name = 'Error')
Error_models_df_clean = Error_models_df[~Error_models_df['Error Method'].str.startswith('mse')]
plt.figure( figsize = (8,5))
ax = sns.barplot( data = Error_models_df_clean , x = 'Error Method' , y = 'Error' , hue = 'Method')
ax.set_title('Combined Error')
plt.legend(loc = 'lower right')
for container in ax.containers:
    ax.bar_label( container)
plt.tight_layout()

On the validation set, the LGBM model performs exceptionally well, with MAE and RMSE scores significantly lower than those of the Hybrid model. However, on the test set, the two models appear quite evenly matched, with no substantial difference in performance. Overall, the LGBM model holds a slight edge over the Hybrid model, though not by a wide margin.

Comparing the accuracy of two machine learning models.

In [ ]:
measure_hd_df = pd.DataFrame( measure_hd , index = ['Hybrid'])
measure_lgb_df = pd.DataFrame( measure , index = ['LGBM'])
Accuracy_df = pd.concat([measure_lgb_df , measure_hd_df])
Accuracy_df['Method'] = Accuracy_df.index
Accuracy_df = Accuracy_df.melt( id_vars = 'Method' , var_name = 'Accuracy Method' , value_name = 'Accuracy')
plt.figure( figsize = (8 , 5))
ax = sns.barplot( data = Accuracy_df , x = 'Accuracy Method' , y = 'Accuracy' , hue = 'Method')
ax.set_title('Combined Accuracy')
plt.legend(loc = 'lower right')
for container in ax.containers:
    ax.bar_label( container)
plt.tight_layout()

Although both models yield similar test accuracy, the LGBM model demonstrates far greater stability across datasets, avoiding the performance gap observed in the hybrid model.

Compare the distribution of residuals for the two models on the validation set.

In [ ]:
fig , axes = plt.subplots( nrows = 2 , ncols = 1 , figsize = (12,8) , sharex = True)
axes.flatten()
sns.histplot( data = y_valid_hd - y_pred_valid_hd , kde = True , label='Hybrid Model', color='orange', alpha=0.5 , ax = axes[0])
sns.histplot( data = y_valid - my_pre_val , kde = True , label = 'LGBM Model' , color = 'blue' , alpha = 0.5 , ax = axes[0] )
axes[0].legend(loc = 'lower right')
axes[0].set_title('residual plots of the two models on the validition set')
sns.histplot( data = y_test_hd - y_pred_test_hd , kde = True , label='Hybrid Model', color='orange', alpha=0.5 , ax = axes[1])
sns.histplot( data = y_test - my_pre_test , kde = True , label = 'LGBM Model' , color = 'blue' , alpha = 0.5 , ax = axes[1] )
axes[1].legend(loc = 'lower right')
axes[1].set_title('residual plots of the two models on the test set')
plt.tight_layout()

Observations show that both models exhibit a bell-shaped distribution of residuals centered at zero, indicating that both are performing very well. However, the LGBM model slightly outperforms the hybrid model in terms of error control, the hybrid model's residual distribution is more dispersed compared to that of the LGBM model.

Compare the dispersion of residuals for the two models on the validation set.

In [ ]:
fig, axes = plt.subplots( nrows = 2 , ncols = 1 , figsize = (12,8) , sharex = True)
axes.flatten()
axes[0].scatter( x = y_valid_hd, y = y_valid_hd - y_pred_valid_hd, label='Hybrid Model', color='green', alpha=0.5 )
axes[0].scatter( x = y_valid, y = y_valid - my_pre_val, label='LGMB Model', color='blue', alpha=0.5 )
axes[1].scatter( x = y_test_hd, y = y_test_hd - y_pred_test_hd, label='Hybrid Model', color='green')
axes[1].scatter( x = y_test, y = y_test - my_pre_test, label='LGMB Model', color='blue', alpha=0.5 )
axes[0].legend(loc = 'lower right')
axes[0].axhline(0, color='red', linestyle='--', linewidth=2)
axes[0].set_xlabel('Actual Revenue')
axes[0].set_ylabel('margin of error')
axes[0].set_title('Compare the remainders: LGBM vs Hybrid(Validition Set)')
axes[1].legend(loc = 'lower right')
axes[1].axhline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Actual Revenue')
axes[1].set_ylabel('margin of error')
axes[1].set_title('Compare the remainders: LGBM vs Hybrid(Test Set)')

The LGBM model controls errors better than the hybrid model on the validation set. Regarding the test set, the two models perform comparably, particularly in that both tend to struggle with periods of sudden, sharp revenue spikes. Nevertheless, the LGBM model still outperforms the hybrid model due to its consistency across datasets.

**4. General conclusion**

The evaluation indicates that the LGBM model outperforms the hybrid method in terms of accuracy, error minimization, and overall stability. However, this does not render the hybrid model obsolete; model selection depends heavily on the specific characteristics of the dataset. Ultimately, this reinforces a key analytical principle: one should avoid deploying overly complex architectures for relatively simple datasets. A notable similarity between the two models is their relative stability when predicting low-to-medium revenue ranges, contrasted with their inability to handle unusually high revenue thresholds. This does not imply that the models are inherently weak when dealing with volatile input data; rather, it suggests they have not sufficiently learned features associated with data volatility. Consequently, it is necessary to consider incorporating additional features related to such fluctuations—such as public holidays or major vacation periods.